# Tokenizadores

In [2]:
from tokenizers import Tokenizer
import torch

In [3]:
tokenizador = Tokenizer.from_file("./data/tokenizer.json")
texto = "O gato dorme"

tokens = [5, 8, 2, 9, 4, 7]
context_size = 3
stride = 1

In [4]:
resultado = tokenizador.encode(texto)
# ['O', 'Ġga', 'to', 'Ġdor', 'me'] => Ġ representa espaço
print(resultado.tokens)
print(resultado.ids)

['O', 'Ġga', 'to', 'Ġdor', 'me']
[47, 4326, 287, 4956, 547]


In [5]:
texto_reconstruido = tokenizador.decode(resultado.ids)
print(texto_reconstruido)
print(texto_reconstruido == texto)

O gato dorme
True


# Ids

In [6]:
entrada = resultado.ids[:-1]
alvo = resultado.ids[1:]

print(entrada, alvo)

[47, 4326, 287, 4956] [4326, 287, 4956, 547]


# Stride

In [7]:
for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x = tokens[i: i + context_size]
    y = tokens[i + 1: i + context_size + 1]
    print(x, y)

i =  0
[5, 8, 2] [8, 2, 9]
i =  1
[8, 2, 9] [2, 9, 4]
i =  2
[2, 9, 4] [9, 4, 7]


# Tensores

In [8]:
x = []
y = []

for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x.append(tokens[i: i + context_size])
    y.append(tokens[i + 1: i + context_size + 1])
    print(x, y)
    
tensor_x = torch.tensor(x, dtype=torch.long)
tensor_y = torch.tensor(y, dtype=torch.long)
print(tensor_x, tensor_y)
print(tensor_x.shape)
print(tensor_y.shape)

i =  0
[[5, 8, 2]] [[8, 2, 9]]
i =  1
[[5, 8, 2], [8, 2, 9]] [[8, 2, 9], [2, 9, 4]]
i =  2
[[5, 8, 2], [8, 2, 9], [2, 9, 4]] [[8, 2, 9], [2, 9, 4], [9, 4, 7]]
tensor([[5, 8, 2],
        [8, 2, 9],
        [2, 9, 4]]) tensor([[8, 2, 9],
        [2, 9, 4],
        [9, 4, 7]])
torch.Size([3, 3])
torch.Size([3, 3])


# Batch

In [9]:
batch_size = 2


def batching(tensor_list, size=batch_size):
    batch = []
    for i in range(0, len(tensor_list), size):
        batch.append(tensor_list[i : i + size])

    return batch


batched_x = batching(tensor_x)
batched_y = batching(tensor_y)

print(batched_x, batched_y)
print(len(batched_x), len(batched_y))

[tensor([[5, 8, 2],
        [8, 2, 9]]), tensor([[2, 9, 4]])] [tensor([[8, 2, 9],
        [2, 9, 4]]), tensor([[9, 4, 7]])]
2 2


# Embeddings

In [10]:
num_embeddings = 10
num_dimensions = 4

embeddings = torch.nn.Embedding(num_embeddings, num_dimensions)

In [11]:
token_vectors = embeddings(batched_x[0])

print(token_vectors)
print("Tabela:", embeddings.weight.shape)
print("Entrada:", batched_x[0].shape)
print("Resultado:", token_vectors.shape)
print(embeddings.weight[8])

tensor([[[ 0.3886, -0.1954,  0.8882,  0.6361],
         [-0.6447,  1.1483,  0.3992, -1.3313],
         [ 1.0183,  0.9530,  0.5584, -1.3304]],

        [[-0.6447,  1.1483,  0.3992, -1.3313],
         [ 1.0183,  0.9530,  0.5584, -1.3304],
         [-0.3251, -0.0711,  0.1949,  0.3026]]], grad_fn=<EmbeddingBackward0>)
Tabela: torch.Size([10, 4])
Entrada: torch.Size([2, 3])
Resultado: torch.Size([2, 3, 4])
tensor([-0.6447,  1.1483,  0.3992, -1.3313], grad_fn=<SelectBackward0>)


# Posição

In [12]:
num_linhas = 3
num_componentes = 4

posicao = torch.nn.Embedding(num_linhas, num_componentes)
value = torch.arange(num_linhas)

In [13]:
position_vectors = posicao(value)
print(position_vectors, position_vectors.shape)

tensor([[-0.3403, -0.0425, -0.1173, -0.1630],
        [-0.9673,  1.0061, -0.0122, -1.2042],
        [-1.0433,  0.8722,  2.4725,  0.6522]], grad_fn=<EmbeddingBackward0>) torch.Size([3, 4])


In [14]:
input_vectors = position_vectors + token_vectors
print(input_vectors, input_vectors.shape)
print("Tokens:", token_vectors)
print("Posições:", position_vectors)

tensor([[[ 0.0483, -0.2379,  0.7709,  0.4731],
         [-1.6120,  2.1545,  0.3870, -2.5355],
         [-0.0250,  1.8252,  3.0309, -0.6783]],

        [[-0.9850,  1.1058,  0.2819, -1.4943],
         [ 0.0509,  1.9592,  0.5462, -2.5347],
         [-1.3684,  0.8011,  2.6674,  0.9547]]], grad_fn=<AddBackward0>) torch.Size([2, 3, 4])
Tokens: tensor([[[ 0.3886, -0.1954,  0.8882,  0.6361],
         [-0.6447,  1.1483,  0.3992, -1.3313],
         [ 1.0183,  0.9530,  0.5584, -1.3304]],

        [[-0.6447,  1.1483,  0.3992, -1.3313],
         [ 1.0183,  0.9530,  0.5584, -1.3304],
         [-0.3251, -0.0711,  0.1949,  0.3026]]], grad_fn=<EmbeddingBackward0>)
Posições: tensor([[-0.3403, -0.0425, -0.1173, -0.1630],
        [-0.9673,  1.0061, -0.0122, -1.2042],
        [-1.0433,  0.8722,  2.4725,  0.6522]], grad_fn=<EmbeddingBackward0>)


# Atenção

In [15]:
camadas_lineares = [
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
]

chamadas = []

for linear in camadas_lineares:
    chamadas.append(linear(input_vectors))
    
queries, keys, values = chamadas
chamadas = []

print(queries, keys, values)

tensor([[[ 0.0629, -0.1948,  0.0854,  0.3838],
         [ 1.9584, -1.3072,  1.1800, -0.8298],
         [ 1.9024, -1.3277,  1.7608,  1.0645]],

        [[ 1.1279, -0.8167,  0.6421, -0.4816],
         [ 1.5141, -0.7333,  1.2873, -0.1313],
         [ 1.3467, -1.3494,  0.9162,  0.6427]]], grad_fn=<UnsafeViewBackward0>) tensor([[[-0.0779, -0.0104, -0.3644,  0.1203],
         [ 1.2454, -1.3720,  0.8529,  1.4157],
         [ 0.5945, -1.7046, -0.3933,  1.2780]],

        [[ 0.7334, -0.7705,  0.4246,  0.8606],
         [ 0.8082, -1.6346,  0.4977,  0.8754],
         [ 0.4357, -0.5043, -0.4423,  1.1813]]], grad_fn=<UnsafeViewBackward0>) tensor([[[-2.5012e-01,  1.7578e-01, -4.4585e-01,  1.3049e-01],
         [ 3.0019e-01, -2.2725e-01, -1.6417e-02,  3.1356e-02],
         [-1.4285e-01,  7.6252e-01, -1.4345e+00,  1.0088e+00]],

        [[ 1.7210e-01, -1.7795e-01, -1.5059e-03,  3.3611e-02],
         [ 9.6205e-01, -4.6835e-02,  3.3651e-01,  8.5697e-01],
         [-1.3026e+00,  6.9092e-01, -1.9973e+00, 

In [16]:
resultado_produto_escalar = []
for i in range(3):
    resultado_produto_escalar.append(sum(queries[0, 1] * keys[0, i]))
    
tensor_pe = torch.stack(resultado_produto_escalar)
half_tensor_pe = tensor_pe/2
print(resultado_produto_escalar)
print(tensor_pe)
print(half_tensor_pe)


[tensor(-0.6688, grad_fn=<AddBackward0>), tensor(4.0642, grad_fn=<AddBackward0>), tensor(1.8680, grad_fn=<AddBackward0>)]
tensor([-0.6688,  4.0642,  1.8680], grad_fn=<StackBackward0>)
tensor([-0.3344,  2.0321,  0.9340], grad_fn=<DivBackward0>)


In [17]:
mascara = half_tensor_pe.clone()
mascara[2] = float("-inf")
print(mascara)

tensor([-0.3344,  2.0321,    -inf], grad_fn=<CopySlices>)


In [18]:
attention_weights = torch.softmax(mascara, dim=0)
print(attention_weights)
print(attention_weights.sum())

tensor([0.0858, 0.9142, 0.0000], grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [19]:
weighted_vectors = []
for i in range(3):
    weighted_vectors.append(attention_weights[i]*values[0, i])
    
weighted_vectors = torch.stack(weighted_vectors)
print(weighted_vectors)
print(weighted_vectors.sum(dim=0))

tensor([[-0.0215,  0.0151, -0.0382,  0.0112],
        [ 0.2744, -0.2078, -0.0150,  0.0287],
        [-0.0000,  0.0000, -0.0000,  0.0000]], grad_fn=<StackBackward0>)
tensor([ 0.2530, -0.1927, -0.0532,  0.0399], grad_fn=<SumBackward1>)


In [23]:
def sliding_mascara(value):
    size = len(value)
    mascaras = []
    for i in range(size):
        mascara = value.clone()
        mascara[i + 1 : size] = float("-inf")
        mascaras.append(torch.softmax(mascara, dim=0))

    return mascaras


weighted_vectors = []
sliding_attention_weights = sliding_mascara(half_tensor_pe)
print(sliding_attention_weights)

[tensor([1., 0., 0.], grad_fn=<SoftmaxBackward0>), tensor([0.0858, 0.9142, 0.0000], grad_fn=<SoftmaxBackward0>), tensor([0.0657, 0.7006, 0.2337], grad_fn=<SoftmaxBackward0>)]


In [24]:
weighted_vectors = []
for i in range(len(sliding_attention_weights)):
    for j in range(len(sliding_attention_weights[i])):
        weighted_vectors.append(sliding_attention_weights[i][j]*values[0, j])
    
weighted_vectors = torch.stack(weighted_vectors)
print(weighted_vectors)
print(weighted_vectors.sum(dim=0))

tensor([[-0.2501,  0.1758, -0.4458,  0.1305],
        [ 0.0000, -0.0000, -0.0000,  0.0000],
        [-0.0000,  0.0000, -0.0000,  0.0000],
        [-0.0215,  0.0151, -0.0382,  0.0112],
        [ 0.2744, -0.2078, -0.0150,  0.0287],
        [-0.0000,  0.0000, -0.0000,  0.0000],
        [-0.0164,  0.0116, -0.0293,  0.0086],
        [ 0.2103, -0.1592, -0.0115,  0.0220],
        [-0.0334,  0.1782, -0.3352,  0.2357]], grad_fn=<StackBackward0>)
tensor([ 0.1634,  0.0136, -0.8751,  0.4366], grad_fn=<SumBackward1>)
